# HerbScan AI - MobileNetV3 Training
Upload your dataset zip file to Colab, then run all cells.

In [ ]:
# 1. Unzip your dataset (Replace 'dataset.zip' with your actual zip file name)
!unzip -q dataset.zip

In [ ]:
import tensorflow as tf
import json
import os
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras import layers, models

# 2. Configuration
# UPDATE THIS PATH to point to where your plant class folders are extracted
DATA_DIR = 'dataset' 
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# 3. Load dataset
train_ds = tf.keras.utils.image_dataset_from_directory(
  DATA_DIR,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=IMG_SIZE,
  batch_size=BATCH_SIZE)

val_ds = tf.keras.utils.image_dataset_from_directory(
  DATA_DIR,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=IMG_SIZE,
  batch_size=BATCH_SIZE)

class_names = train_ds.class_names
print('\nDetected Classes:', class_names)

# 4. Save class names JSON
with open('class_names.json', 'w') as f:
    json.dump(class_names, f)
print('Saved class_names.json!')

In [ ]:
# 5. Build the MobileNetV3 Model
base_model = MobileNetV3Large(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False # Freeze base model for transfer learning

model = models.Sequential([
  base_model,
  layers.GlobalAveragePooling2D(),
  layers.Dropout(0.2),
  layers.Dense(len(class_names), activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 6. Train the Model
print("Starting training...")
history = model.fit(train_ds, validation_data=val_ds, epochs=10)

# 7. Save the final Model
model.save('medicinal_leaf_mobilenetv3.keras')
print('\n✅ Saved model to medicinal_leaf_mobilenetv3.keras. You can now download it from the Colab file explorer!')